# Nexus Local AI v3 — Jupyter-safe version

Runs verification and loads the persistent ledger **without** starting the terminal-only `input()` prompt.

Run the code cell; it prints `NEXUS ENGINE READY`. Use another code cell to call `commute(machine, 5)` or the model via `ask_model`.


In [1]:
"""NEXUS local AI / compiler bridge — reader-relative frame v3.
Dean A. Kulik, QuHarmonics Research Group, ORCID 0009-0003-3128-8828.

Python 3.10+, standard library only. Runs as a script or in one Jupyter cell.
LM Studio: http://localhost:1235/v1 (override with LM_STUDIO_URL).

Conceptual state: (w,E,p) with E[r]=D[p XOR r].
Lazy field: E[r]=V[r XOR s], backing dictionary V and internal frame s.
F: x=V[s]^w; V'[s]=V[s]^P(x); s'=s^x; p'=p^x; w'=P(x).
H_h: p'=p^h, w'=w, (V',s')=(V,s).
Both forward and transport perform O(1) native field operations.
F H_h = H_h F, H_h^(-1)=H_h, H_a H_b=H_(a^b).
Old absolute snapshots and v1/v2 hash-linked ledger records remain readable.
Full ledger before/after snapshots still require O(active cells) serialization.

The local language model requests tools; the compiler computes and verifies operations.
No unrestricted shell or filesystem tool is exposed to the model.
"""
from __future__ import annotations
import hashlib
import json
import os
import random
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

BASE = os.getenv('LM_STUDIO_URL', 'http://localhost:1235/v1').rstrip('/')
if BASE.endswith('/v1'):
    BASE = BASE[:-3]
TOKEN = os.getenv('LM_API_TOKEN', '')
WIDTH, SEED_P, SEED_W = 4, 0, 1
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
LEDGER = ROOT / 'NEXUS_local_AI_events.jsonl'  # Reuse & verify the original history.
GENESIS = '0' * 64


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)


def chain_hash(previous, payload):
    return hashlib.sha256((previous + '\n' + canonical(payload)).encode('utf-8')).hexdigest()


def partner(x: int, n: int) -> int:
    """Native P: reverse nonpalindromes, complement palindromes."""
    mask = (1 << n) - 1
    if not (2 <= n <= 16 and 0 <= x <= mask):
        raise ValueError('n must be 2..16 and x must fit n bits')
    rev = int(f'{x:0{n}b}'[::-1], 2)
    return rev if rev != x else x ^ mask


class Machine:
    """Native conceptual state (w, E, p); lazy E[r]=V[r XOR s].

    V is the sparse backing field, s the internal frame, p the absolute reader WHERE.
    Both forward/backward and transport touch O(1) native field cells.
    Exporting a complete absolute/relative field costs O(k) for k occupied cells.
    """
    def __init__(self, n=WIDTH, p=SEED_P, w=SEED_W, field=None,
                 *, relative_field=None):
        if not (2 <= n <= 16):
            raise ValueError('n must be 2..16 for this prototype')
        mask = (1 << n) - 1
        if type(p) is not int or type(w) is not int or not (0 <= p <= mask and 0 <= w <= mask):
            raise ValueError('p and w must be n-bit words')
        if field is not None and relative_field is not None:
            raise ValueError('supply absolute field or relative_field, not both')
        # Initial frame is zero: the backing keys begin in reader-relative coordinates.
        if relative_field is not None:
            backing = {int(k): int(v) for k, v in relative_field.items() if int(v)}
        else:
            backing = {int(k) ^ p: int(v) for k, v in (field or {}).items() if int(v)}
        if any(not (0 <= k <= mask and 0 <= v <= mask) for k,v in backing.items()):
            raise ValueError('field keys and values must be n-bit words')
        self.n, self.p, self.w = n, p, w
        self.V, self.s = backing, 0
        self.last_hash = GENESIS

    def copy(self):
        other = Machine(self.n, self.p, self.w, relative_field={})
        other.V, other.s = self.V.copy(), self.s
        other.last_hash = self.last_hash
        return other

    def adopt(self, other):
        self.p, self.w = other.p, other.w
        self.V, self.s = other.V.copy(), other.s
        self.last_hash = other.last_hash

    def snapshot(self):
        """External v2-compatible state (p,w,D), for tool and ledger compatibility."""
        absolute = {self.p ^ self.s ^ k: v for k,v in self.V.items()}
        return {'n': self.n, 'p': self.p, 'w': self.w,
                'field': {str(k): absolute[k] for k in sorted(absolute)}}

    def relative_snapshot(self):
        relative = {k ^ self.s: v for k,v in self.V.items()}
        return {str(k): relative[k] for k in sorted(relative)}

    @staticmethod
    def _set(field, at, value):
        if value:
            field[at] = value
        else:
            field.pop(at, None)

    def forward(self):
        """One-cell native step, O(1) independent of field occupancy."""
        oldp, oldframe = self.p, self.s
        x = self.V.get(oldframe, 0) ^ self.w
        w2 = partner(x, self.n)
        self._set(self.V, oldframe, self.V.get(oldframe, 0) ^ w2)
        self.s, self.p, self.w = oldframe ^ x, oldp ^ x, w2
        return {'direction': 'forward', 'x': x, 'written_at': oldp}

    def backward(self):
        """One-cell exact inverse, independent of the number of occupied cells."""
        x = partner(self.w, self.n)
        oldframe, oldp = self.s ^ x, self.p ^ x
        self._set(self.V, oldframe, self.V.get(oldframe, 0) ^ self.w)
        oldw = x ^ self.V.get(oldframe, 0)
        self.s, self.p, self.w = oldframe, oldp, oldw
        return {'direction': 'reverse', 'x': x, 'restored_at': oldp}

    def transport(self, h: int):
        """O(1) pure placement; both V and s remain literally unchanged."""
        if type(h) is not int or not 0 <= h < (1 << self.n):
            raise ValueError(f'transport h must be an integer 0..{(1 << self.n)-1}')
        prev = self.p
        self.p ^= h
        # Keep exact v2 route metadata to preserve historical event-replay checks.
        return {'direction': 'transport', 'h': h, 'from_word': prev,
                'to_word': self.p, 'relocated_field_cells': len(self.V)}

def verify():
    """Executed deterministic checks: involution, random F/H commuting squares, routes."""
    checks = 0
    for n in range(2, 11):
        for x in range(1 << n):
            assert partner(partner(x, n), n) == x
            assert partner(x, n) != x
            checks += 1
    rng = random.Random(181)
    for n in range(2, 17):
        mask = (1 << n) - 1
        for _ in range(320):
            D = {}
            for __ in range(1 + rng.randrange(12)):
                k = rng.randrange(mask + 1)
                v = rng.randrange(mask + 1)
                if v:
                    D[k] = v
            m = Machine(n, rng.randrange(mask+1), rng.randrange(mask+1), D)
            h, a, b = (rng.randrange(mask+1) for __ in range(3))
            # H is its own inverse; transport group is XOR.
            t = m.copy(); original_V = t.V; original_s = t.s
            t.transport(h)
            assert t.V is original_V and t.s == original_s
            t.transport(h)
            assert t.V is original_V and t.s == original_s
            assert t.snapshot() == m.snapshot()
            t = m.copy(); t.transport(a); t.transport(b)
            u = m.copy(); u.transport(a ^ b)
            assert t.snapshot() == u.snapshot()
            # The commuting square F o H == H o F.
            left = m.copy(); left.transport(h); left.forward()
            right = m.copy(); right.forward(); right.transport(h)
            assert left.snapshot() == right.snapshot(), ('FH=HF', n)
            # Also commute for inverse steps.
            left = m.copy(); left.transport(h); left.backward()
            right = m.copy(); right.backward(); right.transport(h)
            assert left.snapshot() == right.snapshot(), ('F^-1H=HF^-1', n)
            # Roundtrip on arbitrary populated field, not just vacuum seed.
            t = m.copy(); t.forward(); t.backward()
            assert t.snapshot() == m.snapshot()
            t = m.copy(); t.backward(); t.forward()
            assert t.snapshot() == m.snapshot()
            checks += 6
    for n in (4, 8, 16):
        m = Machine(n=n, p=0, w=1)
        initial = m.snapshot()
        for _ in range(1000):
            m.forward()
        for _ in range(1000):
            m.backward()
        assert m.snapshot() == initial
    print(f'RUNS: {checks:,} deterministic conditions; P, arbitrary-field roundtrip, '
          'transport involution/group law, F H = H F, inverse commutation, '
          'and 1000-step return at n=4,8,16.')


def apply_mutation(machine, kind, h=None):
    if kind == 'step':
        return machine.forward()
    if kind == 'reverse':
        return machine.backward()
    if kind == 'transport':
        return machine.transport(h)
    raise ValueError('unknown operation')


def replay():
    m = Machine()
    if not LEDGER.exists():
        return m
    count, legacy = 0, 0
    with LEDGER.open('r', encoding='utf-8') as f:
        for count, line in enumerate(f, 1):
            event = json.loads(line)
            if event['before'] != m.snapshot():
                raise RuntimeError(f'Ledger discontinuity before line {count}')
            kind = event['operation']
            # Supports old unhashed events without rewriting or discarding them.
            if 'event_hash' in event:
                if event.get('prev_hash') != m.last_hash:
                    raise RuntimeError(f'Ledger hash link invalid at line {count}')
                payload = {k: v for k, v in event.items()
                           if k not in ('prev_hash', 'event_hash')}
                digest = chain_hash(m.last_hash, payload)
                if digest != event['event_hash']:
                    raise RuntimeError(f'Ledger event hash invalid at line {count}')
            else:
                legacy += 1
                payload = event
                digest = chain_hash(m.last_hash, payload)
            meta = apply_mutation(m, kind, event.get('route', {}).get('h'))
            if event['after'] != m.snapshot() or (event.get('route') and meta != event['route']):
                raise RuntimeError(f'Ledger event mismatch after line {count}')
            m.last_hash = digest
    print(f'Replayed {count} event(s), including {legacy} legacy event(s). '
          f'Last chain digest: {m.last_hash}')
    return m


def operation(m: Machine, kind: str, h=None):
    if kind == 'state':
        return {'status': 'RUNS', 'state': m.snapshot(), 'relative_field': m.relative_snapshot(), 'ledger_head': m.last_hash}
    if kind not in ('step', 'reverse', 'transport'):
        return {'error': 'Only state, step, reverse, transport are allowed'}
    before = m.snapshot()
    trial = m.copy()
    try:
        meta = apply_mutation(trial, kind, h)
    except ValueError as e:
        return {'error': str(e), 'status': 'OPEN'}
    after = trial.snapshot()
    # Independent compiler return; never ask the model to certify it.
    check = trial.copy()
    if kind == 'step':
        check.backward()
    elif kind == 'reverse':
        check.forward()
    else:
        check.transport(h)
    assert check.snapshot() == before, f'{kind}: way home failed'
    payload = {'version': 3, 'operation': kind, 'before': before,
               'after': after, 'route': meta}
    digest = chain_hash(m.last_hash, payload)
    record = {**payload, 'prev_hash': m.last_hash, 'event_hash': digest}
    # Write before committing the in-memory state, so a disk failure cannot create
    # an unrecorded accepted transition. Existing ledger lines are never rewritten.
    with LEDGER.open('a', encoding='utf-8') as f:
        f.write(canonical(record) + '\n')
        f.flush()
        os.fsync(f.fileno())
    trial.last_hash = digest
    m.adopt(trial)
    return {'status': 'RUNS', **record, 'return_verified': True,
            'relative_field': m.relative_snapshot(),
            'native_field_key_rewrites': 0 if kind == 'transport' else 1}


def commute(m: Machine, h: int):
    """Run two commuting paths on copies; leave the live state and ledger unchanged."""
    if type(h) is not int or not 0 <= h < (1 << m.n):
        return {'status': 'OPEN', 'error': 'invalid n-bit XOR offset'}
    starting = m.snapshot()
    left = m.copy(); left.transport(h); left.forward()
    right = m.copy(); right.forward(); right.transport(h)
    assert left.snapshot() == right.snapshot(), 'commuting square failed'
    lback = left.copy(); lback.backward(); lback.transport(h)
    rback = right.copy(); rback.transport(h); rback.backward()
    assert lback.snapshot() == starting == rback.snapshot(), 'commuting return failed'
    return {'status': 'RUNS', 'identity': 'F(H_h(S)) == H_h(F(S))',
            'h': h, 'starting': starting, 'left': left.snapshot(),
            'right': right.snapshot(), 'both_returns_verified': True,
            'live_machine_unchanged': m.snapshot() == starting,
            'ledger_head': m.last_hash, 'ledger_head_unchanged': True}


def request_json(method, endpoint, payload=None, timeout=180):
    headers = {'Content-Type': 'application/json'}
    if TOKEN:
        headers['Authorization'] = f'Bearer {TOKEN}'
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    req = Request(BASE + endpoint, headers=headers, data=data, method=method)
    with urlopen(req, timeout=timeout) as res:
        return json.loads(res.read().decode('utf-8'))


def find_model():
    data = request_json('GET', '/api/v1/models')
    loaded = [(model['display_name'], inst['id'])
              for model in data.get('models', []) if model.get('type') == 'llm'
              for inst in model.get('loaded_instances', [])]
    if not loaded:
        print('No loaded LLM. Load Gemma 4 E4B in LM Studio, then rerun.')
        return None
    for i, (label, ident) in enumerate(loaded, 1):
        print(f'  {i}. {label} [{ident}]')
    print('Using first loaded model:', loaded[0][0])
    return loaded[0][1]


TOOLS = [
    {'type':'function','function':{'name':'nexus_state',
       'description':'Read current exact reader, carry, sparse field, ledger digest.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_step',
       'description':'Execute one exact reversible native partner reader-field step.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_reverse',
       'description':'Execute exact inverse native partner reader-field step.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_transport',
       'description':'Relocate full reader-field state by n-bit XOR offset h. '
         'Preserves behavior: F H_h = H_h F and H_h is its own inverse.',
       'parameters':{'type':'object','properties':{'h':{
           'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1,
           'description':'XOR word offset, integer 0..15 for current n=4'}},
           'required':['h'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_commute',
       'description':'Read-only deterministic verification of F H_h = H_h F on '
                     'the current populated state; verifies both return routes.',
       'parameters':{'type':'object','properties':{'h':{
           'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1}},
           'required':['h'],'additionalProperties':False}}},
]

SYSTEM = ('You interface with an independently executing NEXUS binary compiler. '
          'Use tools for actual machine state or changes. Never invent tool results. '
          'Transport h is an integer XOR word offset; it changes the absolute reader '
          'coordinate while the native relative field remains unchanged. '
          'State clearly which operations actually ran. The compiler owns verification.')


def ask_model(model_id, machine, user_text, history=None):
    # History is optional; if supplied, context persists within this Python session.
    messages = history if history is not None else [{'role':'system','content':SYSTEM}]
    messages.append({'role':'user','content':user_text})
    for _ in range(16):
        response = request_json('POST', '/v1/chat/completions', {
            'model': model_id, 'messages': messages, 'tools': TOOLS,
            'tool_choice':'auto', 'temperature':0.1, 'max_tokens':640,
            'stream':False,
        })
        msg = response['choices'][0]['message']
        calls = msg.get('tool_calls') or []
        if not calls:
            content = msg.get('content') or '(no text returned)'
            messages.append({'role':'assistant','content':content})
            print('\nMODEL:', content)
            return
        messages.append({'role':'assistant','content':msg.get('content'),
                         'tool_calls':calls})
        for call in calls:
            fn = call.get('function',{}).get('name','')
            kind = {'nexus_state':'state','nexus_step':'step',
                    'nexus_reverse':'reverse','nexus_transport':'transport',
                    'nexus_commute':'commute'}.get(fn,'')
            try:
                args = json.loads(call.get('function',{}).get('arguments') or '{}')
                if not isinstance(args,dict):
                    raise ValueError('arguments must be a JSON object')
                if kind in ('transport','commute') and 'h' not in args:
                    raise ValueError('nexus_transport requires h')
                result = commute(machine, args['h']) if kind == 'commute' else operation(machine, kind, args.get('h'))
            except (ValueError, TypeError, json.JSONDecodeError) as err:
                result = {'status':'OPEN','error':str(err)}
            print(f'  TOOL {fn}:', canonical(result))
            messages.append({'role':'tool','tool_call_id':call['id'],
                             'content':canonical(result)})
    print('Model tool-round budget reached; exact machine state:', canonical(machine.snapshot()))


def main():
    verify()
    machine = replay()
    print('Current state:', canonical(machine.snapshot()))
    print('LM Studio:', BASE)
    try:
        model_id = find_model()
    except (URLError, HTTPError, TimeoutError, ValueError) as err:
        print('LM Studio not reachable:', err)
        model_id = None
    chat = [{'role':'system','content':SYSTEM}]
    print('Commands: :state, :step, :reverse, :transport <h>, :commute <h>, :clear, :quit')
    while True:
        try:
            line = input('nexus> ').strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not line: continue
        if line == ':quit': break
        if line == ':clear':
            chat[:] = [{'role':'system','content':SYSTEM}]
            print('Conversation context reset; deterministic machine unchanged.')
        elif line in (':state',':step',':reverse'):
            print(json.dumps(operation(machine,line[1:]),indent=2))
        elif line.startswith(':transport') or line.startswith(':commute'):
            parts = line.split()
            if len(parts) != 2:
                print('Usage: :transport 3 or :commute 3  (also accepts 0x3)')
                continue
            try:
                h = int(parts[1],0)
                print(json.dumps(commute(machine,h) if parts[0] == ':commute'
                                 else operation(machine,'transport',h),indent=2))
            except ValueError as err:
                print('Invalid transport:', err)
        elif not model_id:
            print('Model unavailable; compiler commands still work.')
        else:
            try:
                ask_model(model_id,machine,line,chat)
            except (URLError,HTTPError,TimeoutError,KeyError,ValueError) as err:
                print('LM Studio API error:',err)


# Jupyter entrypoint: initialize without a blocking terminal input() loop.
verify()
machine = replay()
print('\nNEXUS ENGINE READY')
print('Current state:', canonical(machine.snapshot()))
print('LM Studio API:', BASE)
print('Run another notebook cell to call:')
print('  operation(machine, "state")')
print('  commute(machine, 5)')
print('  operation(machine, "step")')
print('  operation(machine, "reverse")')
print('  operation(machine, "transport", 5)')
print('  model_id = find_model()')
print('  ask_model(model_id, machine, "Read state and verify commute h=5",')
print('            [{"role":"system", "content": SYSTEM}])')


RUNS: 30,844 deterministic conditions; P, arbitrary-field roundtrip, transport involution/group law, F H = H F, inverse commutation, and 1000-step return at n=4,8,16.
Replayed 19 event(s), including 4 legacy event(s). Last chain digest: d3c28760a59dc2aa6f4de437de31d217e0e9e976af823b3c183a047cb1513077

NEXUS ENGINE READY
Current state: {"field":{"4":1,"5":8},"n":4,"p":12,"w":1}
LM Studio API: http://localhost:1235
Run another notebook cell to call:
  operation(machine, "state")
  commute(machine, 5)
  operation(machine, "step")
  operation(machine, "reverse")
  operation(machine, "transport", 5)
  model_id = find_model()
  ask_model(model_id, machine, "Read state and verify commute h=5",
            [{"role":"system", "content": SYSTEM}])


In [2]:

model_id = find_model()

if model_id:
    ask_model(
        model_id,
        machine,
        """Use the Nexus compiler tools to:
        1. Read the current machine state.
        2. Verify F H_5 = H_5 F using nexus_commute.
        3. Report both computed paths and their endpoints.
        4. Verify that the original machine state is unchanged.

        Use actual tool results, not simulated calculations.""",
        [{"role": "system", "content": SYSTEM}]
    )
else:
    print("No model loaded in LM Studio.")


  1. Gemma 4 E4B [google/gemma-4-e4b]
Using first loaded model: Gemma 4 E4B
  TOOL nexus_state: {"ledger_head":"d3c28760a59dc2aa6f4de437de31d217e0e9e976af823b3c183a047cb1513077","relative_field":{"8":1,"9":8},"state":{"field":{"4":1,"5":8},"n":4,"p":12,"w":1},"status":"RUNS"}
  TOOL nexus_commute: {"both_returns_verified":true,"h":5,"identity":"F(H_h(S)) == H_h(F(S))","ledger_head":"d3c28760a59dc2aa6f4de437de31d217e0e9e976af823b3c183a047cb1513077","ledger_head_unchanged":true,"left":{"field":{"0":8,"1":1,"9":8},"n":4,"p":8,"w":8},"live_machine_unchanged":true,"right":{"field":{"0":8,"1":1,"9":8},"n":4,"p":8,"w":8},"starting":{"field":{"4":1,"5":8},"n":4,"p":12,"w":1},"status":"RUNS"}

MODEL: (no text returned)


In [3]:

import json

proof = commute(machine, 5)

response = request_json("POST", "/v1/chat/completions", {
    "model": model_id,
    "messages": [
        {
            "role": "system",
            "content": "Explain verified compiler results in plain text. Do not invent results."
        },
        {
            "role": "user",
            "content": (
                "Explain in three sentences what this NEXUS "
                "commuting-square result establishes:\n"
                + json.dumps(proof)
            )
        }
    ],
    "temperature": 0.1,
    "max_tokens": 1536,
    "stream": False
})

choice = response["choices"][0]
print("FINISH:", choice.get("finish_reason"))
print("MODEL:", choice["message"].get("content") or "(empty)")


FINISH: stop
MODEL: This result verifies a mathematical identity, specifically that two operations, represented by functions $F$ and $H$, commute when applied sequentially. The core finding is that applying function $F$ first, followed by $H$ on the result ($F(H(S))$), yields the exact same outcome as applying $H$ first, followed by $F$ on the original input ($H(F(S))$). Because both sides of this equation were verified and the status is "RUNS," the compiler confirms that the order in which these two operations are performed does not change the final result.
